In [1]:
# ===============================
# 1. IMPORT LIBRARIES
# ===============================
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# ===============================
# 2. DEFINE PROJECT PATHS
# ===============================
BASE_PATH = Path.cwd().parent

DATA_PATH = BASE_PATH / "dataset"
MODELS_PATH = BASE_PATH / "models"

print("Data Path:", DATA_PATH)
print("Models Path:", MODELS_PATH)

Data Path: c:\Users\venut\OneDrive\Desktop\job_AI_system\dataset
Models Path: c:\Users\venut\OneDrive\Desktop\job_AI_system\models


In [3]:
# ===============================
# 3. LOAD ANALYTICS DATASET
# ===============================
analytics_df = pd.read_csv(DATA_PATH / "analytics_jobs.csv")

print("Dataset Shape:", analytics_df.shape)
analytics_df.head()

Dataset Shape: (7522, 16)


,uniq_id,job_title,industry,industry_group,role,role_group,industry_role,city_cleaned,city_group,salary_disclosed,salary_min,salary_max,salary_avg,exp_avg,exp_bucket,key_skills_cleaned
0,7b921f51b5c2fb862b4a5f7a54c37f75,Technical Support,"it-software, software services","it-software, software services",Technical Support Engineer,Tech,"it-software, software services_Tech",Mumbai,Mumbai,1,200000.0,400000.0,300000.0,2.5,Mid,technical support
1,c9628ea8d9cfd2ca72e14d53783aca90,Opening For Adobe Analytics Specialist,"it-software, software services","it-software, software services",System Analyst,Other,"it-software, software services_Other",Pune,Pune,1,700000.0,1700000.0,1200000.0,6.0,Senior,"adobe experience manager, digital, digital mar..."
2,2008eb365fa0ebcb1ccd9a50efc3da49,Sales- Fresher-for Leading Property Consultant,"real estate, property",Other,Sales Executive/Officer,Sales,Other_Sales,Bengaluru,Bengaluru,1,200000.0,300000.0,250000.0,0.0,Entry,"channel partners, real estate, negotiation, pr..."
3,d3fac027fd78efae29c1f9394a6cd576,Looking Facebook /social Media Manager For ou...,"advertising, pr, mr, event management",Other,Social Media Marketing Manager,Operations,Other_Operations,Delhi,Delhi,1,350000.0,500000.0,425000.0,3.0,Mid,"digital marketing, seo, social media marketing..."
4,4740f30cacc84382a30c6fc714dcfa94,Job Openings Kotak Life/ Max Life/ Aditya Bir...,insurance,insurance,Sales Officer,Sales,insurance_Sales,Bhilai/Bhillai,Other,1,100000.0,325000.0,212500.0,5.0,Mid,"sales manager, business development manager, r..."


In [4]:
# ===============================
# 4. LOAD TRAINED ARTIFACTS
# ===============================
salary_model = joblib.load(MODELS_PATH / "salary_model.pkl")
tfidf = joblib.load(MODELS_PATH / "tfidf_vectorizer.pkl")
medians = joblib.load(MODELS_PATH / "medians.pkl")

print("Artifacts loaded successfully.")

Artifacts loaded successfully.


In [5]:
# ===============================
# 5. CREATE JOB TF-IDF MATRIX
# ===============================
job_tfidf_matrix = tfidf.transform(
    analytics_df["key_skills_cleaned"].fillna("")
)

print("TF-IDF Matrix Shape:", job_tfidf_matrix.shape)

TF-IDF Matrix Shape: (7522, 300)


In [6]:
# ===============================
# 6. DEFINE USER PROFILE
# ===============================
user_skills = "python sql machine learning pandas"
user_exp = 3  # years of experience

print("User Skills:", user_skills)
print("User Experience:", user_exp)

User Skills: python sql machine learning pandas
User Experience: 3


In [7]:
# ===============================
# 7. PREPARE USER FEATURES (FIXED VERSION)
# ===============================

# Get feature names used during training
feature_names = salary_model.feature_names_in_

# Create a zero-filled dataframe with same structure
user_df = pd.DataFrame(
    np.zeros((1, len(feature_names))),
    columns=feature_names
)

# Fill numeric features if present
if "exp_avg" in user_df.columns:
    user_df.loc[0, "exp_avg"] = user_exp

if "exp_squared" in user_df.columns:
    user_df.loc[0, "exp_squared"] = user_exp ** 2

# Create experience bucket
def create_exp_bucket(exp):
    if exp <= 2:
        return "Entry"
    elif exp <= 5:
        return "Mid"
    elif exp <= 10:
        return "Senior"
    else:
        return "Leadership"

exp_bucket = create_exp_bucket(user_exp)

bucket_col = f"exp_bucket_{exp_bucket}"
if bucket_col in user_df.columns:
    user_df.loc[0, bucket_col] = 1

# ---------------------------
# TF-IDF FEATURES
# ---------------------------
user_tfidf = tfidf.transform([user_skills])

tfidf_feature_names = tfidf.get_feature_names_out()

for i, word in enumerate(tfidf_feature_names):
    if word in user_df.columns:
        user_df.loc[0, word] = user_tfidf[0, i]

print("User feature vector aligned with training model.")

User feature vector aligned with training model.


In [8]:
# ===============================
# 8. PREDICT USER SALARY
# ===============================

pred_log_salary = salary_model.predict(user_df)[0]
predicted_salary = np.expm1(pred_log_salary)

print("Predicted Suitable Salary:", round(predicted_salary, 2))

Predicted Suitable Salary: 641139.6


In [9]:
# ===============================
# 9. SKILL SIMILARITY
# ===============================

user_tfidf = tfidf.transform([user_skills])

skill_similarity = cosine_similarity(
    user_tfidf,
    job_tfidf_matrix
).flatten()

analytics_df["skill_score"] = skill_similarity

In [10]:
# ===============================
# 10. SALARY MATCH SCORE
# ===============================

def salary_score(job_salary):
    if pd.isna(job_salary):
        return 0.3  # small penalty if not disclosed
    
    return 1 - abs(job_salary - predicted_salary) / predicted_salary

analytics_df["salary_score"] = analytics_df["salary_avg"].apply(salary_score)

In [11]:
# ===============================
# 11. EXPERIENCE MATCH SCORE
# ===============================

max_exp = analytics_df["exp_avg"].max()

analytics_df["exp_score"] = 1 - (
    abs(analytics_df["exp_avg"] - user_exp) / max_exp
)

In [12]:
# ===============================
# 12. FINAL HYBRID SCORE
# ===============================

analytics_df["final_score"] = (
    0.5 * analytics_df["skill_score"] +
    0.3 * analytics_df["salary_score"] +
    0.2 * analytics_df["exp_score"]
)

In [13]:
# ===============================
# 13. TOP 10 RECOMMENDATIONS
# ===============================

top_jobs = analytics_df.sort_values(
    by="final_score",
    ascending=False
).head(10)

top_jobs[[
    "job_title",
    "industry_group",
    "role_group",
    "city_group",
    "salary_avg",
    "exp_avg",
    "final_score"
]]

,job_title,industry_group,role_group,city_group,salary_avg,exp_avg,final_score
4928,C Python App/Production Support Pune Intervie...,"it-software, software services",Tech,Pune,650000.0,3.0,0.857583
840,Immediate Opening Forpython with C++ - CMML 5...,"it-software, software services",Tech,Bengaluru,550000.0,5.5,0.824704
1723,Trainee-data Scientist,"it-software, software services",Other,Hyderabad,825000.0,2.5,0.797319
5768,"Hiring For Data Scraping (python,beautiful So...","it-software, software services",Other,Delhi,450000.0,4.5,0.785913
5472,Urgent Hiring For Database Programmer- Gurgaon,"bpo, call centre, ites",Tech,Delhi,600000.0,4.5,0.784914
2919,Data Engineer,"it-software, software services",Tech,Delhi,600000.0,2.5,0.780388
6248,Tableau Developer II Viman Nagar II 20 Days Max,Other,Tech,Pune,550000.0,4.5,0.761518
6719,Artifical Intelligence / Machine Learning Dev...,Other,Tech,Delhi,600000.0,4.5,0.758360
3481,Python Developer/salary Up to 9 Lpa/noida,"it-software, software services",Tech,Delhi,650000.0,4.0,0.757424
1121,opening for Data Scientist_Hyderabad,"it-software, software services",Tech,Hyderabad,1150000.0,4.0,0.753896
